# GitLab Projects Export

### What's new
- **Bug fix**: `@ubs.websdk/` imports now correctly detected and shown in `library`, `import_statement`, `import_file_url`
- **Sample mode**: test individual projects before running the full export
- **Performance**: projects without `package.json` skip the entire file tree scan
- **Outputs**: XLSX + JSON (full run) and sample XLSX + JSON (sample mode)

## 📦 Imports

In [ ]:
import getpass
import json
import logging
import re
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone, timedelta
from typing import Any, Dict, List, Optional, Tuple

import gitlab
import pandas as pd

logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger('gitlab_export')
logger.info('Ready.')

## ⚙️ Configuration

In [ ]:
GITLAB_URL   = 'https://devcloud.ubs.net'
GROUP_PATH   = 'ubs/gwma'
OUTPUT_XLSX  = 'gitlab_projects_export.xlsx'
OUTPUT_JSON  = 'gitlab_projects_export.json'
SAMPLE_XLSX  = 'sample_gitlab_projects_export.xlsx'
SAMPLE_JSON  = 'sample_gitlab_projects_export.json'
PER_PAGE     = 100
MAX_WORKERS  = 20   # parallel project threads
FILE_WORKERS = 10   # parallel file-fetch threads per project
CUTOFF_DAYS  = 365  # only process projects created within this many days

# ---- Set to False to skip sample prompts and run full export directly ----
ENABLE_SAMPLE_MODE = True

SOURCE_EXTENSIONS = ('.jsx', '.tsx', '.js', '.ts')

# BUG FIX: Previously @ubs\.websdk pattern had an escaped dot which caused
# mismatches. Pattern now uses a literal dot inside the character class and
# re.IGNORECASE so component name casing never blocks a match.
IMPORT_RE = re.compile(
    r"^[ \t]*import\s+\{[^}]+\}(?:\s*,\s*\{[^}]+\})*\s+from\s+"
    r"['\"](@ubs\.websdk/[^'\"]+|@uwr[^'\"]+)['\"]\s*;?[^\n]*",
    re.MULTILINE | re.IGNORECASE,
)

CUTOFF = datetime.now(timezone.utc) - timedelta(days=CUTOFF_DAYS)
logger.info('Cutoff date: %s', CUTOFF.date())
logger.info('Sample mode: %s', ENABLE_SAMPLE_MODE)

## 🔐 Authentication

In [ ]:
private_token = getpass.getpass('Enter your GitLab private token: ')
client = gitlab.Gitlab(GITLAB_URL, private_token=private_token)
logger.info('Client ready for %s', GITLAB_URL)

## 🔧 Helper Functions

In [ ]:
def created_within(created_at_str: str) -> bool:
    try:
        dt = datetime.fromisoformat(created_at_str.replace('Z', '+00:00'))
        return dt >= CUTOFF
    except Exception:
        return True


def extract_team_name(web_url: str) -> str:
    try:
        segs = web_url.rstrip('/').split('//', 1)[-1].split('/')[1:]
        if len(segs) >= 2:
            return segs[-2]
    except Exception:
        pass
    return ''


def _raw_file(project: Any, path: str, ref: str) -> Optional[str]:
    """Fetch raw file bytes — avoids base64 round-trip."""
    try:
        return project.files.raw(file_path=path, ref=ref).decode('utf-8', errors='replace')
    except Exception:
        return None


def get_package_json_info(project: Any, ref: str) -> Tuple[str, str]:
    """
    Returns ('Yes'/'No', 'internal'/'external'/'-').
    'No' = no package.json → caller must skip file tree scan.
    """
    content = _raw_file(project, 'package.json', ref)
    if content is None:
        logger.info('  [pkg] not found | %s', project.path_with_namespace)
        return 'No', '-'
    try:
        deps = json.loads(content).get('dependencies', {})
    except json.JSONDecodeError as e:
        logger.warning('  [pkg] bad JSON | %s | %s', project.path_with_namespace, e)
        return 'Yes', 'external'
    uwr = [d for d in deps if d.startswith('@uwr/')]
    if uwr:
        logger.info('  [pkg] internal | %s | %s', project.path_with_namespace, uwr)
        return 'Yes', 'internal'
    logger.info('  [pkg] external | %s', project.path_with_namespace)
    return 'Yes', 'external'


def scan_imports(project: Any, ref: str) -> Tuple[str, str, str]:
    """
    Walk repo tree, scan .js/.ts/.jsx/.tsx for @ubs.websdk/ or @uwr/ imports.
    Returns bracket-delimited (filenames, blob-urls, import-statements).
    Only called when package.json exists.
    """
    try:
        all_items = project.repository_tree(ref=ref, recursive=True, all=True, per_page=100)
    except Exception as e:
        logger.warning('  [scan] tree error | %s | %s', project.path_with_namespace, e)
        return '', '', ''

    src = [i for i in all_items
           if i.get('type') == 'blob' and i['path'].endswith(SOURCE_EXTENSIONS)]

    if not src:
        logger.info('  [scan] no source files | %s', project.path_with_namespace)
        return '', '', ''

    logger.info('  [scan] %d source files | %s', len(src), project.path_with_namespace)

    filenames, urls, stmts = [], [], []

    def scan_one(item):
        path = item['path']
        content = _raw_file(project, path, ref)
        if not content:
            return None
        results = []
        for m in IMPORT_RE.finditer(content):
            results.append((
                path.split('/')[-1],
                f"{project.web_url}/-/blob/{ref}/{path}",
                m.group(0).strip(),
            ))
        return results or None

    with ThreadPoolExecutor(max_workers=FILE_WORKERS) as pool:
        for result in pool.map(scan_one, src):
            if result:
                for fn, fu, st in result:
                    filenames.append(fn)
                    urls.append(fu)
                    stmts.append(st)

    if not filenames:
        logger.info('  [scan] no matching imports | %s', project.path_with_namespace)
        return '', '', ''

    logger.info('  [scan] %d import(s) found | %s', len(filenames), project.path_with_namespace)
    fmt = lambda lst: ''.join(f'[{v}]' for v in lst)
    return fmt(filenames), fmt(urls), fmt(stmts)


def derive_library(import_statements: str) -> str:
    """
    Returns '', 'uwr', 'websdk', or 'uwr, websdk' — no duplicates.
    Case-insensitive check.
    """
    if not import_statements:
        return ''
    s = import_statements.lower()
    found = []
    if '@uwr/' in s:
        found.append('uwr')
    if '@ubs.websdk/' in s:
        found.append('websdk')
    return ', '.join(found)


def process_one(proj_ref: Any, idx: int, total: int) -> Optional[Dict[str, Any]]:
    """
    Processing order (fastest checks first):
      1. Date filter     — zero extra API calls
      2. package.json    — one file fetch; returns None if missing (skips tree walk)
      3. File tree scan  — only if package.json exists
    """
    # 1. Date filter
    created_at_str = getattr(proj_ref, 'created_at', None) or ''
    if created_at_str and not created_within(created_at_str):
        logger.info('[%d/%d] SKIP (old) | %s | created=%s',
                    idx, total, proj_ref.path_with_namespace, created_at_str[:10])
        return None

    logger.info('[%d/%d] → %s', idx, total, proj_ref.path_with_namespace)
    project   = client.projects.get(proj_ref.id)
    namespace = project.namespace or {}
    web_url   = project.web_url
    ref       = project.default_branch or 'main'

    team_name         = extract_team_name(web_url)

    # 2. package.json — skip tree scan if missing
    pkg_json, int_ext = get_package_json_info(project, ref)
    if pkg_json == 'No':
        logger.info('[%d/%d] SKIP (no package.json) | %s', idx, total, project.path_with_namespace)
        return None

    # 3. File scan
    imp_fn, imp_url, imp_st = scan_imports(project, ref)
    library = derive_library(imp_st)

    return {
        'project_id':          project.id,
        'name':                project.name,
        'path':                project.path,
        'path_with_namespace': project.path_with_namespace,
        'group_path':          namespace.get('full_path'),
        'web_url':             web_url,
        'description':         project.description,
        'visibility':          project.visibility,
        'archived':            project.archived,
        'created_at':          project.created_at,
        'last_activity_at':    project.last_activity_at,
        'updated_at':          project.updated_at,
        'default_branch':      ref,
        'forks_count':         getattr(project, 'forks_count', None),
        'star_count':          getattr(project, 'star_count', None),
        'open_issues_count':   getattr(project, 'open_issues_count', None),
        'team_name':           team_name,
        'package_json':        pkg_json,
        'int_ext':             int_ext,
        'component':           '',
        'library':             library,
        'import_filename':     imp_fn,
        'import_file_url':     imp_url,
        'import_statement':    imp_st,
    }


logger.info('All helpers defined.')

## 🔧 Output Writers

In [ ]:
def build_dataframe(rows: List[Dict[str, Any]]) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    for col in ('created_at', 'last_activity_at', 'updated_at'):
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], errors='coerce', utc=True).dt.tz_localize(None)
    df.sort_values(['group_path', 'name'], inplace=True, ignore_index=True)
    return df


def write_xlsx(df: pd.DataFrame, path: str) -> None:
    logger.info('Writing XLSX → %s  (%d rows)', path, len(df))
    with pd.ExcelWriter(path, engine='openpyxl') as writer:
        df.to_excel(writer, index=False, sheet_name='projects')
        ws = writer.sheets['projects']
        for col_cells in ws.columns:
            w = max((len(str(c.value)) if c.value is not None else 0) for c in col_cells)
            ws.column_dimensions[col_cells[0].column_letter].width = min(w + 4, 80)
    logger.info('XLSX done: %s', path)


def write_json(rows: List[Dict[str, Any]], path: str) -> None:
    logger.info('Writing JSON → %s  (%d rows)', path, len(rows))
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(rows, f, indent=2, default=str, separators=(',', ': '))
    logger.info('JSON done: %s', path)


logger.info('Writers defined.')

## 🧪 Sample Mode
Set `ENABLE_SAMPLE_MODE = True` in the config cell above to activate.
Enter project names one at a time; each result is written to the sample files immediately.
Answer **no** when asked for another sample to proceed to the full export.

In [ ]:
sample_rows: List[Dict[str, Any]] = []

if ENABLE_SAMPLE_MODE:
    print('\n' + '='*60)
    print('SAMPLE MODE — test individual projects before full run')
    print('='*60)

    while True:
        project_name = input('\nEnter project name to sample: ').strip()
        if not project_name:
            print('No name entered — skipping.')
        else:
            logger.info('[sample] Searching for: %s', project_name)
            try:
                group     = client.groups.get(GROUP_PATH)
                proj_refs = group.projects.list(
                    search=project_name, include_subgroups=True, all=True, per_page=50
                )
                match = next(
                    (p for p in proj_refs if p.name.lower() == project_name.lower()),
                    proj_refs[0] if proj_refs else None,
                )
            except Exception as e:
                logger.error('[sample] Search failed: %s', e)
                match = None

            if not match:
                print(f'⚠️  No project found matching "{project_name}".')
            else:
                logger.info('[sample] Found: %s', match.path_with_namespace)
                row = process_one(match, 1, 1)
                if row:
                    sample_rows.append(row)
                    print(f'\n✅  {row["path_with_namespace"]}')
                    print(f'    package_json : {row["package_json"]}')
                    print(f'    int_ext      : {row["int_ext"]}')
                    print(f'    library      : {row["library"]}')
                    n_imp = len(row['import_filename'].split('][')) if row['import_filename'] else 0
                    print(f'    imports found: {n_imp}')
                    # Write sample files after every addition
                    df_s = build_dataframe(sample_rows)
                    write_xlsx(df_s, SAMPLE_XLSX)
                    write_json(sample_rows, SAMPLE_JSON)
                    print(f'    Sample files updated: {SAMPLE_XLSX} | {SAMPLE_JSON}')
                else:
                    print(f'⚠️  Project was filtered out (old date or no package.json): {match.path_with_namespace}')

        again = input('\nRun another sample? (yes/no): ').strip().lower()
        if again not in ('yes', 'y'):
            break

    print(f'\nSample mode done — {len(sample_rows)} project(s) in sample files.')
    cont = input('Proceed with full export? (yes/no): ').strip().lower()
    if cont not in ('yes', 'y'):
        print('Stopping here. Run the cells below manually when ready.')
else:
    print('Sample mode disabled — proceeding directly to full export.')

## 🚀 Full Export — Fetch Project List

In [ ]:
logger.info("Fetching project list for group '%s'", GROUP_PATH)
group     = client.groups.get(GROUP_PATH)
proj_refs = group.projects.list(include_subgroups=True, all=True, per_page=PER_PAGE)
total     = len(proj_refs)
logger.info('Total projects in group: %d', total)

## ⚡ Process Projects in Parallel

In [ ]:
rows: List[Dict[str, Any]] = []
done = 0

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
    future_map = {
        pool.submit(process_one, ref, idx, total): ref
        for idx, ref in enumerate(proj_refs, start=1)
    }
    for future in as_completed(future_map):
        done += 1
        ref = future_map[future]
        try:
            row = future.result()
            if row:
                rows.append(row)
                logger.info('[%d/%d ✓] %s', done, total, row['path_with_namespace'])
            else:
                logger.info('[%d/%d –] skipped | %s', done, total, ref.path_with_namespace)
        except Exception as e:
            logger.error('[%d/%d ✗] %s | %s', done, total, ref.path_with_namespace, e)

logger.info('Done: %d included, %d skipped/failed out of %d total', len(rows), total - len(rows), total)

## 👀 Preview

In [ ]:
df = build_dataframe(rows)

print(f'Projects included : {len(df)}')
print(f'With imports found: {(df["import_filename"] != "").sum()}')
if 'library' in df.columns:
    print('\nLibrary breakdown:')
    print(df[df['library'] != '']['library'].value_counts().to_string())

df[['name', 'created_at', 'team_name', 'package_json', 'int_ext', 'library', 'import_filename']].head(10)

## 💾 Write XLSX + JSON

In [ ]:
if not rows:
    logger.warning('No rows to write.')
else:
    write_xlsx(df, OUTPUT_XLSX)
    write_json(rows, OUTPUT_JSON)
    print(f'✅ XLSX → {OUTPUT_XLSX}')
    print(f'✅ JSON → {OUTPUT_JSON}')